In [ ]:
### 8.2.1. Implementation of the ResNet Basic Block

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

## GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")


## ResNet 기본 블록 정의
class BasicBlock(nn.Module):
  expansion = 1

  def __init__(self, in_channels, out_channels, stride=1, downsample=None):
    super(BasicBlock, self).__init__() # nn.Module 상속 → 메서드/함수/속성 사용
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                           stride=stride, padding=1, bias=False) # 합성곱층 1
    self.bn1 = nn.BatchNorm2d(out_channels) # 배치 정규화층 1
    self.relu = nn.ReLU(inplace=True)
    # inplace → 새로운 텐서를 생성 X. 입력 텐서 값을 직접 덮어써서 수정
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                           padding=1, bias=False) # 합성곱층 2
    self.bn2 = nn.BatchNorm2d(out_channels) # 배치 정규화층 2
    self.downsample = downsample
    self.stride = stride

  def forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)


    out = self.conv2(out)
    out = self.bn2(out)

    '''
    downsample 역할
    → 입력 데이터와 잔차 블록의 출력 데이터 크기(차원)를 일치!
    '''
    if self.downsample is not None:
      identity = self.downsample(x)

    out += identity # Skip Connection(잔차 연결)
    out = self.relu(out)

    return out # End of Class BasicBlock(nn.Module)


## 테스트 코드
def test_basic_block():
  # 샘플 입력 생성
  x = torch.randn(1, 64, 32, 32).to(device) # (N, C, H, W)

  # BasicBlock 생성(다운샘플링 X)
  block1 = BasicBlock(64, 64).to(device)
  output1 = block1(x)
  print(f"동일 채널 블록 출력 크기: {output1.shape}")

  # downsample 구성 시: 입력 채널(64)과 출력 채널(128)이 다르므로,
  # 1x1 Conv를 통해 채널 수를 128로 늘리고 가로세로 크기를 조절한다.
  # 이후 BatchNorm을 적용한다.
  downsample = nn.Sequential(
      nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False),
      nn.BatchNorm2d(128)
  ).to(device)

  block2 = BasicBlock(64, 128, stride=2, downsample=downsample).to(device)
  output2 = block2(x)
  print(f"다운샘플링 블록 출력 크기: {output2.shape}")


test_basic_block()

사용 중인 디바이스: cuda
동일 채널 블록 출력 크기: torch.Size([1, 64, 32, 32])
다운샘플링 블록 출력 크기: torch.Size([1, 128, 16, 16])


In [4]:
### 8.2.2.Implementation of the full ResNet model

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

## GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")


## ResNet 기본 블록 정의
class BasicBlock(nn.Module):
  expansion = 1

  def __init__(self, in_channels, out_channels, stride=1, downsample=None):
    super(BasicBlock, self).__init__()
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                           stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                           padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.downsample = downsample
    self.stride = stride

  def forward(self, x):
    identity = x
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    if self.downsample is not None:
      identity = self.downsample(x)

    out += identity # Skip Connection(잔차 연결)
    out = self.relu(out)

    return out # End of Class BasicBlock(nn.Module)


## 전체 ResNet 모델 구현
class ResNet(nn.Module):
  '''
  하이퍼파라미터 설명:
  1. block: 네트워크를 구성하는 기본 빌딩 블록 종류를 지정
  2. layers: ResNet 4개 스테이지(layer1~4)에 몇 개 블록을 쌓는가?
  3. num_classes: 마지막 완전 연결 계층(fc) 출력 노드 개수
  4. zero_init_residual: 잔차 분기 최종 배치 정규화 가중치를 0으로 초기화할지 결정
  '''
  def __init__(self, block, layers, num_classes=1000, zero_init_residual=False):
    super(ResNet, self).__init__()
    self.in_channels = 64

    # 초기 합성곱 층
    self.conv1 = nn.Conv2d(3, self.in_channels, kernel_size=7, stride=2,
                           padding=3, bias=False)
    self.bn1 = nn.BatchNorm2d(self.in_channels)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    # ResNet 블록들
    # remind: 일반적인 ResNet 핵심 구조는 총 4개의 스테이지로 구성된다!
    self.layer1 = self._make_layer(block, 64, layers[0])
    self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
    self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
    self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

    # 분류를 위한 완전 연결층
    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(512 * block.expansion, num_classes)

    # 가중치 초기화
    for m in self.modules():
      # if - 변수 m이 2D 합성곱 신경망 레이어(nn.Conv2d)의 인스턴스인가?
      if isinstance(m, nn.Conv2d):
        # 카이밍 초기화 - 기울기 소실 & 폭주를 막기 위해 가중치를 초기화
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

      elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
        # 정규화 후 채널별 스케일을 1로 → 초기 입력 분포를 왜곡하지 않고 통과
        nn.init.constant_(m.weight, 1)
        # 정규화 후 이동을 0으로 설정하여 초기 편향 제거
        nn.init.constant_(m.bias, 0)

  def _make_layer(self, block, out_channels, blocks, stride=1):
    downsample = None # 필요할 때만 다운샘플링 적용 → 기본값 = None 설정

    # Skip Connection의 차원 조정이 필요한 경우:
    # 보폭이 1이 아니거나, 입력 채널이 보정된 출력 채널과 같지 않을 때
    if stride != 1 or self.in_channels != out_channels * block.expansion:
      downsample = nn.Sequential(
          # 잔차 연결 전후 입력 채널과 출력 채널을 맞춰주는 다운샘플링!
          nn.Conv2d(self.in_channels, out_channels * block.expansion,
                    kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels * block.expansion)
      )

    layers = []
    # 첫번째 레이어는 다운샘플링 수행 가능
    layers.append(block(self.in_channels, out_channels, stride, downsample))
    # 입력 채널 수 업데이트
    self.in_channels = out_channels * block.expansion
    # 나머지 블록 추가(다운샘플링 X)
    for _ in range(1, blocks):
      layers.append(block(self.in_channels, out_channels))
    return nn.Sequential(*layers) # End of def _make_layer

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.maxpool(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)

    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)

    return x
  # End of class ResNet


## ResNet-18 모델 정의
def resnet18(num_classes=10):
  # ResNet-18 모델
  return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

def resnet34(num_classes=10):
  # ResNet-34 모델
  return ResNet(BasicBlock, [3, 4, 6, 3], num_classes=num_classes)


## ResNet-50, 101, 152용 Bottleneck 블록 정의
class Bottleneck(nn.Module):
    expansion = 4 # Bottleneck 블록에서는 채널이 최종 4배 확장된다.

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
      super(Bottleneck, self).__init__()

      # 1×1 conv(차원 축소)
      self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
      self.bn1 = nn.BatchNorm2d(out_channels)

      # 3×3 conv(주요 연산)
      self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                            stride=stride, padding=1, bias=False)
      self.bn2 = nn.BatchNorm2d(out_channels)

      # 1×1 conv(차원 확장)
      self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion,
                            kernel_size=1, bias=False)
      self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

      self.relu = nn.ReLU(inplace=True)
      self.downsample = downsample
      self.stride = stride

    def forward(self, x):
      identity = x

      # 1×1 conv
      out = self.conv1(x)
      out = self.bn1(out)
      out = self.relu(out)

      # 3×3 conv
      out = self.conv2(out)
      out = self.bn2(out)
      out = self.relu(out)

      # 1×1 conv(확장)
      out = self.conv3(out)
      out = self.bn3(out)

      # Skip Connection
      if self.downsample is not None:
        identity = self.downsample(x)

      out += identity
      out = self.relu(out)

      return out ## End of class Bottleneck


## 추가 ResNet 모델 정의

def resnet50(num_classes=10):
  # ResNet-50 모델
  return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes)

def resnet101(num_classes=10):
  # ResNet-101 모델
  return ResNet(Bottleneck, [3, 4, 23, 3], num_classes=num_classes)

def resnet152(num_classes=10):
  # ResNet-152 모델
  return ResNet(Bottleneck, [3, 8, 36, 3], num_classes=num_classes)


## 확장된 테스트 코드

def test_all_resnet():
  # 모든 ResNet 모델 테스트
  models = {
      'ResNet-18': resnet18(num_classes=10),
      'ResNet-34': resnet34(num_classes=10),
      'ResNet-50': resnet50(num_classes=10),
      'ResNet-101': resnet101(num_classes=10),
      'ResNet-152': resnet152(num_classes=10)
  }

  for name, model in models.items():
    model = model.to(device)
    # remind: (1, 3, 224, 224) → PyTorch ResNet 모델의 가장 기본적인 입력 크기
    x = torch.rand(1, 3, 224, 224).to(device)

    # 기울기 업데이트 없이 예측 결과 & 파라미터 수 관찰
    with torch.no_grad():
      output = model(x)
      param_count = sum(p.numel() for p in model.parameters())

    print(f"{name}")
    print(f"  출력 크기: {output.shape}")
    print(f"  파라미터 수: {param_count}")
    print(f"  메모리 사용량: {param_count * 4 / 1024 / 1024: .1f} MB")
    print()

test_all_resnet()

사용 중인 디바이스: cuda
ResNet-18
  출력 크기: torch.Size([1, 10])
  파라미터 수: 11181642
  메모리 사용량:  42.7 MB

ResNet-34
  출력 크기: torch.Size([1, 10])
  파라미터 수: 21289802
  메모리 사용량:  81.2 MB

ResNet-50
  출력 크기: torch.Size([1, 10])
  파라미터 수: 23528522
  메모리 사용량:  89.8 MB

ResNet-101
  출력 크기: torch.Size([1, 10])
  파라미터 수: 42520650
  메모리 사용량:  162.2 MB

ResNet-152
  출력 크기: torch.Size([1, 10])
  파라미터 수: 58164298
  메모리 사용량:  221.9 MB

